## Data Analysis Setup

In this step, we import the required libraries to handle data, perform numerical operations, and connect to a SQL database.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

## Creating Retail Dataset

Here we generate a synthetic retail dataset including customer transactions, products, pricing, and order dates. This simulates real-world business data.

In [2]:
np.random.seed(42)

n_orders = 10000

data = pd.DataFrame({
    "order_id": range(1, n_orders + 1),
    "customer_id": np.random.randint(1, 2000, n_orders),
    "product": np.random.choice(["Laptop","Phone","Tablet","Headphones"], n_orders),
    "price": np.random.choice([50,100,300,800], n_orders),
    "quantity": np.random.randint(1,5, n_orders),
    "order_date": pd.date_range(start="2023-01-01", periods=n_orders, freq="h")
})

data["total_amount"] = data["price"] * data["quantity"]

data.head()

,order_id,customer_id,product,price,quantity,order_date,total_amount
0,1,1127,Tablet,50,1,2023-01-01 00:00:00,50
1,2,1460,Phone,800,2,2023-01-01 01:00:00,1600
2,3,861,Headphones,800,1,2023-01-01 02:00:00,800
3,4,1295,Laptop,300,1,2023-01-01 03:00:00,300
4,5,1131,Headphones,100,2,2023-01-01 04:00:00,200


## Creating SQL Database

In this step, we create a SQLite database and store the dataset as a table called 'orders'. This allows us to run SQL queries on the data.

In [3]:
conn = sqlite3.connect("retail.db")

data.to_sql("orders", conn, if_exists="replace", index=False)

10000

## Previewing Data with SQL

We run a simple SQL query to preview the first rows of the dataset and confirm that the data was correctly stored

In [4]:
query = "SELECT * FROM orders LIMIT 5;"
pd.read_sql(query, conn)

,order_id,customer_id,product,price,quantity,order_date,total_amount
0,1,1127,Tablet,50,1,2023-01-01 00:00:00,50
1,2,1460,Phone,800,2,2023-01-01 01:00:00,1600
2,3,861,Headphones,800,1,2023-01-01 02:00:00,800
3,4,1295,Laptop,300,1,2023-01-01 03:00:00,300
4,5,1131,Headphones,100,2,2023-01-01 04:00:00,200


## Total Revenue Analysis

This query calculates the total revenue generated by the business.

In [5]:
query = """
SELECT SUM(total_amount) AS total_revenue
FROM orders;
"""

pd.read_sql(query, conn)

,total_revenue
0,7883900


## Revenue by Product

This analysis shows how much revenue each product generates, helping identify top-performing products.

In [12]:
query = """
SELECT product, SUM(total_amount) AS revenue
FROM orders
GROUP BY product
ORDER BY revenue DESC;
"""

pd.read_sql(query, conn)

,product,revenue
0,Laptop,2046900
1,Headphones,1964150
2,Tablet,1940650
3,Phone,1932200


## Top Customers by Spending

We identify the customers who have spent the most money, highlighting high-value customers.

In [13]:
query = """
SELECT customer_id, SUM(total_amount) AS total_spent
FROM orders
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_id,total_spent
0,1760,15900
1,1807,14900
2,1261,14700
3,283,14600
4,554,14450
5,936,14300
6,699,13300
7,559,13100
8,1185,12950
9,1939,12800


## Customer Purchase Frequency

This query identifies the most active customers based on the number of orders placed.

In [14]:
query = """
SELECT customer_id, COUNT(order_id) AS total_orders
FROM orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_id,total_orders
0,1953,14
1,1807,13
2,661,13
3,1939,12
4,1800,12
5,1463,12
6,1185,12
7,1063,12
8,947,12
9,936,12


## Average Order Value

This metric calculates the average amount spent per transaction.

In [9]:
query = """
SELECT AVG(total_amount) AS avg_order_value
FROM orders;
"""

pd.read_sql(query, conn)

,avg_order_value
0,788.39


## Monthly Revenue Trend

This analysis shows how revenue changes over time, aggregated by month.

In [10]:
query = """
SELECT 
    substr(order_date, 1, 7) AS month,
    SUM(total_amount) AS revenue
FROM orders
GROUP BY month
ORDER BY month;
"""

pd.read_sql(query, conn)

,month,revenue
0,2023-01,580400
1,2023-02,524550
2,2023-03,585750
3,2023-04,579300
4,2023-05,598350
5,2023-06,544700
6,2023-07,599950
7,2023-08,574000
8,2023-09,538450
9,2023-10,589150


## High-Value Customers (VIP)

We identify customers who have spent more than a defined threshold, representing high-value segments.

In [11]:
query = """
SELECT customer_id, SUM(total_amount) AS total_spent
FROM orders
GROUP BY customer_id
HAVING total_spent > 2000
ORDER BY total_spent DESC;
"""

pd.read_sql(query, conn)

,customer_id,total_spent
0,1760,15900
1,1807,14900
2,1261,14700
3,283,14600
4,554,14450
...,...,...
1446,1116,2050
1447,1052,2050
1448,884,2050
1449,560,2050


## Business Insights

- The business generates significant revenue driven by key products.
- A small group of customers contributes a large portion of total revenue.
- Customer behavior varies in frequency and spending patterns.
- Monthly trends can help identify seasonality and growth opportunities.